# VoiceOfBank — 03 Preprocessing
**Notebook 3 of 7** — Run locally

Clean and prepare 30,000 reviews for NLP modelling.

**Input:** `data/raw/reviews_raw.csv`

**Output:**
- `data/processed/reviews_clean.csv`
- `data/processed/reviews_complaint.csv`
- `data/processed/tfidf_matrix.npz`
- `data/processed/tfidf_vocab.json`
- `data/processed/label_encoder.json`


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import json
import scipy.sparse
import warnings
from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

for resource in ['stopwords','wordnet','punkt','punkt_tab','omw-1.4']:
    nltk.download(resource, quiet=True)

RAW_PATH      = Path('../data/raw/reviews_raw.csv')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BANKS      = ['Monzo','Starling','Barclays','HSBC','NatWest','Lloyds']
CHALLENGER = ['Monzo','Starling']
RANDOM_SEED = 42



## 2. Load Raw Data

In [2]:
df = pd.read_csv(RAW_PATH, parse_dates=['at'])

before = len(df)
df = df.dropna(subset=['content']).reset_index(drop=True)
print(f'Dropped {before - len(df)} null reviews')
print(f'Working with {len(df):,} reviews')

df = df[['reviewId','content','score','at','replyContent','bank']].copy()
df.rename(columns={'content':'text','score':'rating','at':'date'}, inplace=True)

df['type']         = df['bank'].apply(lambda x: 'Challenger' if x in CHALLENGER else 'Traditional')
df['has_reply']    = df['replyContent'].notna().astype(int)
df['review_len']   = df['text'].str.len()
df['year_month']   = df['date'].dt.to_period('M').astype(str)
df['is_complaint'] = (df['rating'] <= 2).astype(int)

print()
print('Columns:', list(df.columns))
df.head(3)


Dropped 1 null reviews
Working with 29,999 reviews

Columns: ['reviewId', 'text', 'rating', 'date', 'replyContent', 'bank', 'type', 'has_reply', 'review_len', 'year_month', 'is_complaint']


,reviewId,text,rating,date,replyContent,bank,type,has_reply,review_len,year_month,is_complaint
0,de2e33fa-8b9f-4d25-b850-ccde652be126,amazing app,5,2026-06-02 09:45:48,NaN,Monzo,Challenger,0,11,2026-06,0
1,8b63c112-adc9-4102-a10d-c5177e111cb2,love monzo it's so easy to use few people said...,5,2026-06-02 08:25:55,NaN,Monzo,Challenger,0,79,2026-06,0
2,d658f4cd-ed2d-40a4-92be-5f86d3ab9555,easy to use the app doesn't seem to crash like...,4,2026-06-01 20:49:59,NaN,Monzo,Challenger,0,71,2026-06,0


## 3. Text Cleaning

Six steps applied in order:

1. Lowercase
2. Remove URLs, emails, version strings
3. Remove punctuation and special characters
4. Normalise whitespace
5. Tokenise
6. Remove stopwords and lemmatise

Custom banking stopwords added — words like 'app', 'bank', 'account'
appear in every review and carry no discriminative signal.


In [3]:
STOP_WORDS = set(stopwords.words('english'))

CUSTOM_STOPS = {
    'app','bank','banking','account','money','use','used','using',
    'barclays','monzo','starling','hsbc','natwest','lloyds',
    'mobile','phone','card','would','could','also','one','get',
    'got','like','really','even','still','good','great','well',
    'time','year','month','day','week','go','going','gone',
}
STOP_WORDS.update(CUSTOM_STOPS)

lemmatizer = WordNetLemmatizer()


def clean_text(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|\S+@\S+', ' ', text)
    text = re.sub(r'v?\d+\.\d+\.?\d*', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in STOP_WORDS and len(t) >= 3
    ]
    return ' '.join(tokens)


sample = df['text'].iloc[0]
print('Original :', sample[:200])
print('Cleaned  :', clean_text(sample)[:200])


Original : amazing app
Cleaned  : amazing


### 3.1 Apply Cleaning

In [4]:
df['text_clean'] = df['text'].apply(clean_text)

before = len(df)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)
print(f'Dropped {before - len(df)} empty reviews after cleaning')
print(f'Remaining: {len(df):,} reviews')

df['clean_len'] = df['text_clean'].str.len()
print()
print('Average cleaned length (chars):')
print(df.groupby('bank')['clean_len'].mean().round(0).sort_values(ascending=False).to_string())
ratio = (df['clean_len'] / df['review_len']).mean()
print(f'\nCompression: {ratio:.1%} of original length retained')


Dropped 2857 empty reviews after cleaning
Remaining: 27,142 reviews

Average cleaned length (chars):
bank
HSBC        60.0
NatWest     50.0
Monzo       49.0
Starling    39.0
Barclays    36.0
Lloyds      31.0

Compression: 55.7% of original length retained


## 4. Sentiment Labels

In [5]:
def rating_to_sentiment(r):
    if r <= 2: return 'Negative'
    if r == 3: return 'Neutral'
    return 'Positive'

df['sentiment'] = df['rating'].apply(rating_to_sentiment)

print('Sentiment distribution:')
counts = df['sentiment'].value_counts()
for cls in ['Positive','Neutral','Negative']:
    n   = counts.get(cls, 0)
    pct = n / len(df) * 100
    print(f'  {cls:<10}: {n:>6,}  ({pct:.1f}%)')

print()
print('Sentiment by bank (% Negative):')
neg_rate = df.groupby('bank').apply(
    lambda x: (x['sentiment']=='Negative').mean()*100
).sort_values(ascending=False).round(1)
print(neg_rate.to_string())


Sentiment distribution:
  Positive  : 19,962  (73.5%)
  Neutral   :    979  (3.6%)
  Negative  :  6,201  (22.8%)

Sentiment by bank (% Negative):
bank
HSBC        44.5
NatWest     36.6
Monzo       25.8
Barclays    11.0
Lloyds      10.8
Starling     8.1


## 5. Label Encoding

In [6]:
bank_to_id = {bank: i for i, bank in enumerate(sorted(df['bank'].unique()))}
id_to_bank = {v: k for k, v in bank_to_id.items()}
df['bank_id'] = df['bank'].map(bank_to_id)

sent_to_id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id_to_sent = {v: k for k, v in sent_to_id.items()}
df['sentiment_id'] = df['sentiment'].map(sent_to_id)

encoder = {
    'bank_to_id': bank_to_id,
    'id_to_bank': {str(k): v for k, v in id_to_bank.items()},
    'sent_to_id': sent_to_id,
    'id_to_sent': {str(k): v for k, v in id_to_sent.items()},
}
with open(PROCESSED_DIR / 'label_encoder.json', 'w') as f:
    json.dump(encoder, f, indent=2)

print('Bank encoding  :', bank_to_id)
print('Sentiment enc  :', sent_to_id)
print('Saved: label_encoder.json')


Bank encoding  : {'Barclays': 0, 'HSBC': 1, 'Lloyds': 2, 'Monzo': 3, 'NatWest': 4, 'Starling': 5}
Sentiment enc  : {'Negative': 0, 'Neutral': 1, 'Positive': 2}
Saved: label_encoder.json


## 6. TF-IDF Matrix

TF-IDF feature matrix for classical ML classifiers in notebook 07.

- `max_features=20000` — keeps vocabulary manageable
- `ngram_range=(1,2)` — captures bigrams like 'customer service', 'wait time'
- `min_df=5` — removes rare terms (fewer than 5 reviews)
- `sublinear_tf=True` — log-scale term frequency dampens very frequent terms


In [7]:
print('Building TF-IDF matrix...')

tfidf = TfidfVectorizer(
    max_features = 20000,
    ngram_range  = (1, 2),
    min_df       = 5,
    max_df       = 0.95,
    sublinear_tf = True,
)

X_tfidf = tfidf.fit_transform(df['text_clean'])

print(f'TF-IDF matrix shape : {X_tfidf.shape}')
print(f'Vocabulary size     : {len(tfidf.vocabulary_):,}')
print(f'Sparsity            : {1 - X_tfidf.nnz/(X_tfidf.shape[0]*X_tfidf.shape[1]):.1%}')

scipy.sparse.save_npz(PROCESSED_DIR / 'tfidf_matrix.npz', X_tfidf)

vocab = {word: int(idx) for word, idx in tfidf.vocabulary_.items()}
with open(PROCESSED_DIR / 'tfidf_vocab.json', 'w') as f:
    json.dump(vocab, f)

print()
print('Top 20 terms by document frequency:')
feature_names = tfidf.get_feature_names_out()
doc_freq = (X_tfidf > 0).sum(axis=0).A1
top_idx  = doc_freq.argsort()[::-1][:20]
for idx in top_idx:
    print(f'  {feature_names[idx]:<25}: {doc_freq[idx]:,} docs')

print('\nSaved: tfidf_matrix.npz, tfidf_vocab.json')


Building TF-IDF matrix...
TF-IDF matrix shape : (27142, 5382)
Vocabulary size     : 5,382
Sparsity            : 99.9%

Top 20 terms by document frequency:
  easy                     : 7,118 docs
  service                  : 2,205 docs
  excellent                : 1,912 docs
  work                     : 1,630 docs
  best                     : 1,332 docs
  need                     : 1,155 docs
  update                   : 1,102 docs
  love                     : 1,095 docs
  customer                 : 1,047 docs
  payment                  : 929 docs
  always                   : 886 docs
  new                      : 864 docs
  make                     : 855 docs
  problem                  : 854 docs
  brilliant                : 843 docs
  issue                    : 843 docs
  keep                     : 834 docs
  quick                    : 824 docs
  simple                   : 810 docs
  year                     : 762 docs

Saved: tfidf_matrix.npz, tfidf_vocab.json


## 7. Train / Test Split

In [8]:
train_idx, test_idx = train_test_split(
    df.index,
    test_size    = 0.20,
    stratify     = df['sentiment'],
    random_state = RANDOM_SEED,
)

df['split'] = 'train'
df.loc[test_idx, 'split'] = 'test'

print('Train/test split (stratified on sentiment):')
print(f'  Train : {(df["split"]=="train").sum():,}')
print(f'  Test  : {(df["split"]=="test").sum():,}')
print()
print('Sentiment distribution in splits:')
print(
    df.groupby('split')['sentiment']
      .value_counts(normalize=True)
      .mul(100).round(1)
      .to_string()
)


Train/test split (stratified on sentiment):
  Train : 21,713
  Test  : 5,429

Sentiment distribution in splits:
split  sentiment
test   Positive     73.5
       Negative     22.8
       Neutral       3.6
train  Positive     73.5
       Negative     22.8
       Neutral       3.6


## 8. Save

In [9]:
df.to_csv(PROCESSED_DIR / 'reviews_clean.csv', index=False)
print('Saved: reviews_clean.csv')
print(f'  Shape  : {df.shape}')
print(f'  Columns: {list(df.columns)}')

complaints = df[df['is_complaint']==1].reset_index(drop=True)
complaints.to_csv(PROCESSED_DIR / 'reviews_complaint.csv', index=False)
print()
print('Saved: reviews_complaint.csv')
print(f'  Shape  : {complaints.shape}')
print()
print('Complaint reviews per bank:')
print(complaints['bank'].value_counts().to_string())
print()
print('All processed files:')
for f in sorted(PROCESSED_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'  {f.name:<35}: {size:.1f} KB')


Saved: reviews_clean.csv
  Shape  : (27142, 17)
  Columns: ['reviewId', 'text', 'rating', 'date', 'replyContent', 'bank', 'type', 'has_reply', 'review_len', 'year_month', 'is_complaint', 'text_clean', 'clean_len', 'sentiment', 'bank_id', 'sentiment_id', 'split']

Saved: reviews_complaint.csv
  Shape  : (6201, 17)

Complaint reviews per bank:
bank
HSBC        2017
NatWest     1662
Monzo       1180
Barclays     499
Lloyds       472
Starling     371

All processed files:
  label_encoder.json                 : 0.4 KB
  reviews_clean.csv                  : 8582.0 KB
  reviews_complaint.csv              : 3357.0 KB
  tfidf_matrix.npz                   : 1663.5 KB
  tfidf_vocab.json                   : 101.7 KB
